In [1]:
import os
import gc
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import altair as alt

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')

print("Imports complete")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\graph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports complete
PyTorch Version: 2.7.1+cu118
CUDA Available: True
GPU Device: NVIDIA GeForce RTX 3060
GPU Memory: 12.88 GB


In [2]:
try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

# Root paths
if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")

Running on Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset


In [3]:
train = pd.read_parquet(f"{DATASET_PATH}/merged_train.parquet")
test = pd.read_parquet(f"{DATASET_PATH}/merged_test.parquet")

RANDOM_SEED = 42
START_DATE = "2026-01-01"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
X = train.drop(columns=["isFraud"])
y = train["isFraud"].values

x = torch.tensor(X.values, dtype=torch.float)
y = torch.tensor(y, dtype=torch.long)

In [5]:
uid_counts = train["uid"].value_counts()

print(uid_counts.describe())
print("Largest uid group:", uid_counts.max())
print(uid_counts.nlargest(20))

count    13657.000000000000000
mean        43.240828878963171
std        324.083169525824928
min          1.000000000000000
25%          1.000000000000000
50%          4.000000000000000
75%         13.000000000000000
max      14112.000000000000000
Name: count, dtype: float64
Largest uid group: 14112
uid
13261    14112
13656    11033
5108     10332
6231     10312
12009     8844
4396      7918
2362      7079
10517     6766
2217      6760
8028      6126
12010     6047
11751     5325
559       5155
2483      5110
8297      4604
7043      4197
4768      3973
5318      3914
8094      3864
5271      3739
Name: count, dtype: int64


In [6]:
K = 5
edge_list = []

for _, group in train.groupby("uid"):
    group = group.sort_values("TransactionDT")
    idx = group.index.to_list()

    n = len(idx)
    if n < 2:
        continue

    for i in range(n):
        for j in range(i + 1, min(i + K + 1, n)):
            edge_list.append([idx[i], idx[j]])
            edge_list.append([idx[j], idx[i]])

edge_index_uid = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(edge_index_uid.shape)

torch.Size([2, 5594098])


In [7]:
K = 5
edge_list = []

for _, group in train.groupby("uid2"):
    group = group.sort_values("TransactionDT")
    idx = group.index.to_list()

    n = len(idx)
    if n < 2:
        continue

    for i in range(n):
        for j in range(i + 1, min(i + K + 1, n)):
            edge_list.append([idx[i], idx[j]])
            edge_list.append([idx[j], idx[i]])

edge_index_uid2 = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(edge_index_uid2.shape)

torch.Size([2, 4610182])


In [8]:
def graph_statistics(edge_index, num_nodes, name):
    degree = torch.bincount(edge_index[0], minlength=num_nodes)

    print(f"===== {name} =====")
    print(f"Nodes           : {num_nodes:,}")
    print(f"Directed edges  : {edge_index.shape[1]:,}")
    print(f"Average degree  : {degree.float().mean():.2f}")
    print(f"Maximum degree  : {degree.max().item():,}")
    print(f"Isolated nodes  : {(degree == 0).sum().item():,}")
    print()

In [9]:
graph_statistics(edge_index_uid, len(train), "G1 (uid)")
graph_statistics(edge_index_uid2, len(train), "G2 (uid2)")

===== G1 (uid) =====
Nodes           : 590,540
Directed edges  : 5,594,098
Average degree  : 9.47
Maximum degree  : 10
Isolated nodes  : 3,473

===== G2 (uid2) =====
Nodes           : 590,540
Directed edges  : 4,610,182
Average degree  : 7.81
Maximum degree  : 10
Isolated nodes  : 36,762



In [10]:
num_nodes = len(train)

indices = np.arange(num_nodes)

train_idx, valid_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=train["isFraud"]
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
valid_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
valid_mask[valid_idx] = True

In [11]:
data_uid = Data(x=x, edge_index=edge_index_uid, y=y)

data_uid.train_mask = train_mask
data_uid.val_mask = valid_mask

In [12]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

        self.classifier = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        return self.classifier(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GraphSAGE(
    in_channels=data_uid.num_node_features,
    hidden_channels=64,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=5e-4,
)

criterion = torch.nn.CrossEntropyLoss()

train_loader = NeighborLoader(
    data_uid,
    input_nodes=data_uid.train_mask,
    num_neighbors=[10, 10],
    batch_size=2048,
    shuffle=True,
)

val_loader = NeighborLoader(
    data_uid,
    input_nodes=data_uid.val_mask,
    num_neighbors=[10, 10],
    batch_size=4096,
    shuffle=False,
)

In [13]:
print("Training configuration: Full-batch approach")
print()

use_neighbor_loader = False

train_indices = torch.where(data_uid.train_mask)[0]
val_indices = torch.where(data_uid.val_mask)[0]

train_dataset = TensorDataset(train_indices)
val_dataset = TensorDataset(val_indices)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2048, shuffle=False)

print(f"Total nodes: {len(data_uid)}")
print(f"Train nodes: {len(train_indices)}")
print(f"Val nodes: {len(val_indices)}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Training configuration: Full-batch approach

Total nodes: 5
Train nodes: 472432
Val nodes: 118108
Train batches: 923
Val batches: 58


In [14]:
def print_memory_stats():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {allocated:.2f} / {total:.2f} GB")

In [ ]:
print(f"Training GraphSAGE (device: {device})")
print()

start_epoch = 1
total_epochs = 20
best_auc = 0
best_epoch = 0

for epoch in range(start_epoch, total_epochs + 1):
    # Train
    model.train()
    total_loss = 0
    num_batches = 0
    
    if use_neighbor_loader:
        # NeighborLoader training (sampling neighborhood)
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            
            # Only the first batch.batch_size nodes are seed nodes
            loss = criterion(out[: batch.batch_size], batch.y[: batch.batch_size])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            num_batches += 1
    else:
        # Fallback full-batch training (using indices)
        for (batch_indices,) in train_loader:
            batch_indices = batch_indices.to(device)
            optimizer.zero_grad()
            
            # Forward pass on full graph
            out = model(data_uid.x.to(device), data_uid.edge_index.to(device))
            
            # Compute loss on batch indices
            loss = criterion(out[batch_indices], data_uid.y[batch_indices].to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            num_batches += 1

    # Validation
    model.eval()
    
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        if use_neighbor_loader:
            # NeighborLoader validation
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index)
                probs = torch.softmax(out[: batch.batch_size], dim=1)[:, 1]
                all_probs.append(probs.cpu())
                all_labels.append(batch.y[: batch.batch_size].cpu())
        else:
            # Fallback validation (single pass on full data)
            out = model(data_uid.x.to(device), data_uid.edge_index.to(device))
            probs = torch.softmax(out, dim=1)[:, 1]
            
            # Get validation indices
            val_indices = torch.where(data_uid.val_mask)[0]
            all_probs = [probs[val_indices].cpu()]
            all_labels = [data_uid.y[val_indices].cpu()]

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    auc = roc_auc_score(labels, probs)
    
    # Track best model
    if auc > best_auc:
        best_auc = auc
        best_epoch = epoch

    print(f"Epoch {epoch:02d}/{total_epochs} | Loss {total_loss/num_batches:.4f} | AUC {auc:.4f}")
    
    if IS_COLAB and epoch % 5 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print()
print(f"Training completed!")
print(f"Best AUC: {best_auc:.4f} (Epoch {best_epoch})")

Training GraphSAGE (device: cuda)



In [ ]:
print("Saving Model and Results")
print()

RESULTS_DIR = SAVED_PATH / "graph_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = f"graphsage_model_{timestamp}"

model_path = RESULTS_DIR / f"{model_name}.pt"
torch.save(model.state_dict(), model_path)
print(f"Model saved: {model_path}")

metadata = {
    "timestamp": timestamp,
    "model_type": "GraphSAGE",
    "training_mode": "full-batch",
    "epochs": 20,
    "hidden_channels": 64,
    "batch_size": 512,
    "device": str(device),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "final_auc": float(auc) if 'auc' in locals() else None,
}

metadata_path = RESULTS_DIR / f"{model_name}_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved: {metadata_path}")